# Comparison Table -- Classification Accuracy + Structural Bucket Breakdown

Cleaner copy of `RQ_4_5/table.ipynb`. The original notebook was a scratchpad with several
superseded drafts of the same scoring logic; this copy keeps only the final, necessary
computation:

1. **Classification accuracy** per output file, split into Overall / ADHD / Control accuracy.
2. **Structural bucket breakdown** for the hub- and community-identification tasks -- each
   response is graded into `correct (>= 0.6)` / `partial (0.4-0.59)` / `suboptimal (< 0.4)` /
   `hallucinatory` / `missing` against ground truth, per model x task x condition.

## 1. Classification accuracy -- Overall / ADHD / Control, per model x file

In [2]:
import os
import json
import pandas as pd

RQ45_DIR = "/home/siu856622573/Code/RQ_4_5"
VALID_LABELS = {"ADHD", "Control"}


def calculate_accuracy(json_path):
    with open(json_path, "r") as f:
        data = json.load(f)

    # Only scorable subjects: valid ground truth AND valid prediction
    scorable = [
        r for r in data.values()
        if r.get("ground_truth") in VALID_LABELS
        and r.get("prediction") in VALID_LABELS
    ]

    overall_correct = sum(r["prediction"] == r["ground_truth"] for r in scorable)
    overall_accuracy = overall_correct / len(scorable) if scorable else None

    adhd = [r for r in scorable if r["ground_truth"] == "ADHD"]
    adhd_correct = sum(r["prediction"] == "ADHD" for r in adhd)
    adhd_accuracy = adhd_correct / len(adhd) if adhd else None

    control = [r for r in scorable if r["ground_truth"] == "Control"]
    control_correct = sum(r["prediction"] == "Control" for r in control)
    control_accuracy = control_correct / len(control) if control else None

    return {
        "Overall Accuracy": overall_accuracy,
        "ADHD Accuracy": adhd_accuracy,
        "Control Accuracy": control_accuracy,
    }


def process_folder(folder_name):
    """Scans RQ45_DIR/<model>/<folder_name>/*.json for every model directory."""
    rows = []
    model_folders = sorted(
        name for name in os.listdir(RQ45_DIR)
        if os.path.isdir(os.path.join(RQ45_DIR, name))
    )

    for model in model_folders:
        target_dir = os.path.join(RQ45_DIR, model, folder_name)
        if not os.path.isdir(target_dir):
            continue

        json_files = sorted(f for f in os.listdir(target_dir) if f.endswith(".json"))
        for json_file in json_files:
            json_path = os.path.join(target_dir, json_file)
            try:
                scores = calculate_accuracy(json_path)
                rows.append({
                    "Model": model,
                    "File": json_file,
                    "Overall Accuracy": scores["Overall Accuracy"],
                    "ADHD Accuracy": scores["ADHD Accuracy"],
                    "Control Accuracy": scores["Control Accuracy"],
                })
            except Exception as e:
                print(f"Error processing: {json_path}\n  {e}")

    return pd.DataFrame(rows)


community_table = process_folder("community")
hub_table = process_folder("hub")

accuracy_columns = ["Overall Accuracy", "ADHD Accuracy", "Control Accuracy"]
for df in [community_table, hub_table]:
    for col in accuracy_columns:
        df[col] = df[col].apply(lambda x: f"{x:.2%}" if pd.notna(x) else "N/A")

print("\n" + "=" * 120)
print("COMMUNITY ACCURACY")
print("=" * 120)
print(community_table.to_string(index=False))

print("\n" + "=" * 120)
print("HUB ACCURACY")
print("=" * 120)
print(hub_table.to_string(index=False))



COMMUNITY ACCURACY
      Model                                                                File Overall Accuracy ADHD Accuracy Control Accuracy
   deepseek                    deepseek_community_for_classification_10com.json           62.11%         3.93%           95.49%
   deepseek            deepseek_community_for_classification_no_edge_10com.json           53.00%        41.73%           59.43%
   deepseek     deepseek_community_for_classification_no_edge_weight_10com.json           61.85%         3.57%           95.29%
   deepseek                                 deepseek_community_gt_36_10com.json           51.10%        41.43%           56.71%
   deepseek                               deepseek_community_gt_36_10com_1.json           51.39%        49.82%           52.28%
deepseek_v4                 deepseek_v4_community_for_classification_10com.json           62.84%         2.50%           97.54%
deepseek_v4         deepseek_v4_community_for_classification_no_edge_10com.json     

## 2. Structural bucket breakdown -- hub & community identification

Grades every prediction against ground truth into one of 5 buckets:
`correct (>= 0.6)` / `partial (0.4-0.59)` / `suboptimal (< 0.4)` / `hallucinatory` / `missing`.

Includes the pair-order-normalization fix (`normalize_pair`) for the community task: ground
truth stores between-network pairs in canonical `network_names` order, but the LLM's free-text
output isn't guaranteed to follow that order (e.g. `"Cont-DorsAttn"` vs. `"DorsAttn-Cont"`) --
without normalizing first, those get misgraded as hallucinatory instead of scored on overlap.

In [3]:
import os
import re
import json
import glob
import pandas as pd

# ─────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────
RQ45_DIR = "/home/siu856622573/Code/RQ_4_5"
CODE_DIR = "/home/siu856622573/Code"

ROI_NAMES_JSON                = os.path.join(CODE_DIR, "aal_roi_names.json")
EDGE_LIST_NAMES_ONLY          = os.path.join(CODE_DIR, "edge_only_from_top10pos_names.json")
NETWORK_NAMES_JSON            = os.path.join(CODE_DIR, "network_names.json")
HUB_GROUND_TRUTH              = os.path.join(CODE_DIR, "hub_ground_truth_20roi.json")
COMMUNITY_GROUND_TRUTH_COUNTS = os.path.join(CODE_DIR, "community_ground_truth_counts.json")

TOP_K_GROUND_TRUTH = 10
VALID_LABELS = {"ADHD", "Control"}

OUTPUT_CSV = os.path.join(RQ45_DIR, "bucket_breakdown.csv")

# ─────────────────────────────────────────────────────────────────
# Loaders
# ─────────────────────────────────────────────────────────────────

def load_roi_names(path):
    with open(path) as f:
        return json.load(f)

def load_brain_graph_texts(path):
    """{subject_id: 'Node feature:\n...\n\nEdge feature:\n...'} -- used to check
    whether a predicted region actually appears in that subject's edges."""
    with open(path) as f:
        return json.load(f)

def load_network_names(path):
    with open(path) as f:
        return json.load(f)

def load_ground_truth_hub_map(path):
    """hub_ground_truth_20roi.json: {subject_id: [{"roi", "degree", "cohort_z"}, ...]}
    Returns {subject_id: [roi_name, ...]} in rank order."""
    with open(path) as f:
        raw = json.load(f)
    return {sid: [entry["roi"] for entry in entries] for sid, entries in raw.items()}

def load_ground_truth_community_map(path, top_n):
    """Merges within_network + between_network dicts into one ranked list of
    'Net-Net' pair strings per subject, top_n highest counts. Keys are already
    stored in canonical order by the ground-truth generator."""
    with open(path) as f:
        raw = json.load(f)
    result = {}
    for sid, record in raw.items():
        combined = {}
        for net, cnt in record["within_network"].items():
            combined[f"{net}-{net}"] = cnt
        for pair, cnt in record["between_network"].items():
            combined[pair] = cnt
        ranked = [k for k, _ in sorted(combined.items(), key=lambda x: x[1], reverse=True)]
        result[sid] = ranked[:top_n]
    return result

def build_all_valid_pairs(network_names):
    """Canonical-order pair names only -- 'a-b' where a comes before b in
    network_names. Any predicted pair must be normalized to this order
    before being checked against this set (see normalize_pair below)."""
    return {f"{n}-{n}" for n in network_names} | {
        f"{a}-{b}" for i, a in enumerate(network_names) for b in network_names[i + 1:]
    }

def build_edge_regions_map(brain_graph_texts):
    return {sid: set(re.findall(r"[A-Za-z0-9_]+", text)) for sid, text in brain_graph_texts.items()}

# ─────────────────────────────────────────────────────────────────
# Pair-order normalization (fixes the "everything reversed = hallucinatory"
# bug -- ground truth stores between-network pairs in canonical
# network_names order, but the LLM's free-text output is not guaranteed to
# follow that order, e.g. "Cont-DorsAttn" instead of "DorsAttn-Cont")
# ─────────────────────────────────────────────────────────────────

def normalize_pair(pair, network_order):
    parts = pair.split("-")
    if len(parts) != 2:
        return pair
    a, b = parts
    if a not in network_order or b not in network_order:
        return pair
    return f"{a}-{b}" if network_order.index(a) <= network_order.index(b) else f"{b}-{a}"

# ─────────────────────────────────────────────────────────────────
# Scoring functions -- ONE rule for both tasks:
#   missing        -> unparseable / empty prediction
#   hallucinatory  -> any predicted item isn't a valid region / pair
#   correct/partial/suboptimal -> overlap ratio vs ground truth top-K
# ─────────────────────────────────────────────────────────────────

def grade_overlap(overlap_ratio):
    if overlap_ratio >= 0.6:
        return "correct (>= 0.6)"
    elif overlap_ratio >= 0.4:
        return "partial (0.4-0.59)"
    else:
        return "suboptimal (< 0.4)"

def score_hub_response(predicted_hubs, ground_truth_top20, all_valid_regions, edge_regions):
    if not predicted_hubs:
        return "missing"

    for region in predicted_hubs:
        if all_valid_regions and region not in all_valid_regions:
            return "hallucinatory"
        if edge_regions and region not in edge_regions:
            return "hallucinatory"

    overlap = len(set(predicted_hubs) & set(ground_truth_top20))
    overlap_ratio = overlap / max(len(predicted_hubs), 1)
    return grade_overlap(overlap_ratio)

def score_community_response(predicted_pairs, gt_top10, all_valid_pairs, network_names):
    if not predicted_pairs:
        return "missing"

    predicted_pairs = [normalize_pair(p, network_names) for p in predicted_pairs]

    for pair in predicted_pairs:
        if pair not in all_valid_pairs:
            return "hallucinatory"

    matches = [p for p in predicted_pairs if p in gt_top10]
    overlap_ratio = len(matches) / max(len(predicted_pairs), 1)
    return grade_overlap(overlap_ratio)

def parse_run_info(path, rq_root):
    """Model / task / condition from filename. Returns None for ground-truth files."""
    rel = os.path.relpath(path, rq_root)
    parts = rel.split(os.sep)
    if len(parts) < 3:
        return None
    model, task = parts[0], parts[1]
    fname = parts[-1].lower()
    if re.search(r"_gt[_.]|_gt$", fname):
        return None  # ground-truth file, not a model run

    if "no_edge_weight" in fname or "edge_no_weight" in fname:
        condition = "no_weight"       # edges present, weights stripped
    elif "no_edge" in fname:
        condition = "no_edge"         # edges stripped entirely
    else:
        condition = "full"
    return model, task, condition

# ─────────────────────────────────────────────────────────────────
# Bucket counting -- same 5 labels for both tasks
# ─────────────────────────────────────────────────────────────────

BUCKETS = [
    "correct (>= 0.6)", "partial (0.4-0.59)", "suboptimal (< 0.4)",
    "hallucinatory", "missing",
]

def hub_bucket_counts(data, ground_truth_hub_map, all_valid_regions, edge_regions_map):
    counts = {b: 0 for b in BUCKETS}
    for sid, r in data.items():
        predicted = r.get("hub_regions") or []
        gt_top20 = ground_truth_hub_map.get(sid, [])
        edge_regions = edge_regions_map.get(sid, set())
        label = score_hub_response(predicted, gt_top20, all_valid_regions, edge_regions)
        counts[label] += 1
    return counts

def community_bucket_counts(data, ground_truth_community_map, all_valid_pairs, network_names):
    counts = {b: 0 for b in BUCKETS}
    for sid, r in data.items():
        predicted = r.get("community_pairs") or []
        gt_top10 = ground_truth_community_map.get(sid, [])
        label = score_community_response(predicted, gt_top10, all_valid_pairs, network_names)
        counts[label] += 1
    return counts

# ─────────────────────────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────────────────────────

def build_bucket_breakdown():
    print("Loading ground truth / resource files...")
    roi_names = load_roi_names(ROI_NAMES_JSON)
    brain_graph_texts = load_brain_graph_texts(EDGE_LIST_NAMES_ONLY)
    edge_regions_map = build_edge_regions_map(brain_graph_texts)
    network_names = load_network_names(NETWORK_NAMES_JSON)

    ground_truth_hub_map = load_ground_truth_hub_map(HUB_GROUND_TRUTH)
    ground_truth_community_map = load_ground_truth_community_map(
        COMMUNITY_GROUND_TRUTH_COUNTS, TOP_K_GROUND_TRUTH
    )
    all_valid_regions = set(roi_names)
    all_valid_pairs = build_all_valid_pairs(network_names)

    print(f"Scanning {RQ45_DIR} for run files...")
    all_files = glob.glob(os.path.join(RQ45_DIR, "**", "*.json"), recursive=True)

    rows = []
    for path in sorted(all_files):
        info = parse_run_info(path, RQ45_DIR)
        if info is None:
            continue
        model, task, condition = info

        with open(path) as f:
            data = json.load(f)

        if task == "hub":
            counts = hub_bucket_counts(data, ground_truth_hub_map, all_valid_regions, edge_regions_map)
        elif task == "community":
            counts = community_bucket_counts(data, ground_truth_community_map, all_valid_pairs, network_names)
        else:
            continue

        row = {"Model": model, "Task": task, "Condition": condition, "N": len(data), "File": os.path.basename(path)}
        row.update(counts)
        rows.append(row)

    df = pd.DataFrame(rows).sort_values(["Task", "Model", "Condition"]).reset_index(drop=True)

    print("\n=== COMMUNITY bucket counts ===")
    comm_df = df[df["Task"] == "community"][["Model", "Condition", "N"] + BUCKETS + ["File"]]
    print(comm_df.to_string(index=False))

    print("\n=== HUB bucket counts ===")
    hub_df = df[df["Task"] == "hub"][["Model", "Condition", "N"] + BUCKETS + ["File"]]
    print(hub_df.to_string(index=False))

    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\nSaved to {OUTPUT_CSV}")
    return df


bucket_df = build_bucket_breakdown()


Loading ground truth / resource files...
Scanning /home/siu856622573/Code/RQ_4_5 for run files...

=== COMMUNITY bucket counts ===
      Model Condition   N  correct (>= 0.6)  partial (0.4-0.59)  suboptimal (< 0.4)  hallucinatory  missing                                                                File
   deepseek      full 768                12                  51                 512              1      192                    deepseek_community_for_classification_10com.json
   deepseek   no_edge 768                15                  51                 515              3      184            deepseek_community_for_classification_no_edge_10com.json
   deepseek no_weight 768                14                  52                 515              3      184     deepseek_community_for_classification_no_edge_weight_10com.json
deepseek_v4      full 768               666                  89                   1             12        0                 deepseek_v4_community_for_classification_

## Peek at the results

In [4]:
print(bucket_df[bucket_df["Task"] == "community"].head(3))
print(bucket_df[bucket_df["Task"] == "hub"].head(3))

with open(HUB_GROUND_TRUTH) as f:
    sample = json.load(f)
first_sid = next(iter(sample))
print(first_sid, sample[first_sid][:3])


      Model       Task  Condition    N  \
0  deepseek  community       full  768   
1  deepseek  community    no_edge  768   
2  deepseek  community  no_weight  768   

                                                File  correct (>= 0.6)  \
0   deepseek_community_for_classification_10com.json                12   
1  deepseek_community_for_classification_no_edge_...                15   
2  deepseek_community_for_classification_no_edge_...                14   

   partial (0.4-0.59)  suboptimal (< 0.4)  hallucinatory  missing  
0                  51                 512              1      192  
1                  51                 515              3      184  
2                  52                 515              3      184  
       Model Task  Condition    N  \
19  deepseek  hub       full  768   
20  deepseek  hub    no_edge  768   
21  deepseek  hub  no_weight  768   

                                                 File  correct (>= 0.6)  \
19         deepseek_hub_for_classifica